# 02 - Model Training

This notebook trains chest X-ray pneumonia classifiers using transfer learning.

**Approach:**
1. Load and augment data with proper train/validation split
2. Use LR Finder to determine optimal learning rate
3. 2-stage fine-tuning: frozen backbone → unfrozen with discriminative LRs
4. Compare ResNet34, ResNet50, and DenseNet121 architectures

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data import create_dataloaders, load_config
from src.model import create_learner, count_parameters
from fastai.vision.all import *

config = load_config('../config/config.yaml')
print('Config loaded successfully')

## 1. Data Loading & Augmentation

In [ ]:
dls = create_dataloaders(
    data_path=config['data']['dataset_path'],
    image_size=config['data']['image_size'],
    batch_size=config['data']['batch_size'],
    valid_pct=config['data']['valid_pct'],
    seed=config['data']['seed'],
)

print(f'Training samples: {len(dls.train_ds)}')
print(f'Validation samples: {len(dls.valid_ds)}')
print(f'Classes: {dls.vocab}')

In [ ]:
# Visualize augmented samples
dls.show_batch(max_n=9, figsize=(10, 10))

## 2. Train ResNet34

In [ ]:
learn_r34 = create_learner(dls, arch_name='resnet34')
print(f'ResNet34 parameters: {count_parameters(learn_r34):,}')

In [ ]:
# Learning Rate Finder
lr_result = learn_r34.lr_find(suggest_funcs=(valley, slide))
print(f'Suggested LR (valley): {lr_result.valley:.2e}')

In [ ]:
# Stage 1: Frozen backbone
learn_r34.fit_one_cycle(3, lr_max=lr_result.valley, wd=0.01)

In [ ]:
# Stage 2: Unfreeze and train with discriminative LRs
learn_r34.unfreeze()
learn_r34.fit_one_cycle(5, lr_max=slice(lr_result.valley/100, lr_result.valley/10), wd=0.01)

In [ ]:
# Save model
learn_r34.export('../outputs/models/resnet34_best.pkl')
print('ResNet34 model saved')

## 3. Train ResNet50

In [ ]:
# Fresh DataLoaders
dls = create_dataloaders(
    data_path=config['data']['dataset_path'],
    image_size=config['data']['image_size'],
    batch_size=config['data']['batch_size'],
    valid_pct=config['data']['valid_pct'],
    seed=config['data']['seed'],
)

learn_r50 = create_learner(dls, arch_name='resnet50')
print(f'ResNet50 parameters: {count_parameters(learn_r50):,}')

In [ ]:
lr_result = learn_r50.lr_find(suggest_funcs=(valley, slide))
print(f'Suggested LR: {lr_result.valley:.2e}')

In [ ]:
learn_r50.fit_one_cycle(3, lr_max=lr_result.valley, wd=0.01)
learn_r50.unfreeze()
learn_r50.fit_one_cycle(5, lr_max=slice(lr_result.valley/100, lr_result.valley/10), wd=0.01)

In [ ]:
learn_r50.export('../outputs/models/resnet50_best.pkl')
print('ResNet50 model saved')

## 4. Train DenseNet121

In [ ]:
dls = create_dataloaders(
    data_path=config['data']['dataset_path'],
    image_size=config['data']['image_size'],
    batch_size=config['data']['batch_size'],
    valid_pct=config['data']['valid_pct'],
    seed=config['data']['seed'],
)

learn_dn = create_learner(dls, arch_name='densenet121')
print(f'DenseNet121 parameters: {count_parameters(learn_dn):,}')

In [ ]:
lr_result = learn_dn.lr_find(suggest_funcs=(valley, slide))
print(f'Suggested LR: {lr_result.valley:.2e}')

In [ ]:
learn_dn.fit_one_cycle(3, lr_max=lr_result.valley, wd=0.01)
learn_dn.unfreeze()
learn_dn.fit_one_cycle(5, lr_max=slice(lr_result.valley/100, lr_result.valley/10), wd=0.01)

In [ ]:
learn_dn.export('../outputs/models/densenet121_best.pkl')
print('DenseNet121 model saved')

## 5. Training Summary

All three architectures have been trained with:
- Proper data augmentation (rotation, horizontal flip, lighting, warp)
- LR Finder for optimal learning rate selection
- 2-stage fine-tuning with discriminative learning rates
- ImageNet normalization

Next: Run `03_evaluation.ipynb` for comprehensive evaluation on the held-out test set.